# # To understand the theory better, visit my GitHub  

<https://github.com/Sakhawathossen04/Building-LLMs-from-scratch-In-Bangla/tree/main/Chapter%206%20%3A%20The%20Tokenizers%20library/6.3%20Fast%20tokenizers%E2%80%99%20special%20powers/All%20Theory>


### This topic has been discussed here in both Bangla and English.

# Special Powers of Fast Tokenizers

In this section, we will look at the extra capabilities of tokenizers in Transformers.

So far, we have mainly used tokenizers for two tasks:
* splitting input text into tokens
* converting token IDs back into text

But tokenizers, especially those built on the Tokenizers library, can do much more.

To explain these extra features, this section shows how to manually reproduce the results of token-classification (NER) and question-answering pipelines.


### What are slow tokenizers and fast tokenizers?
There are two types of tokenizers mentioned here:
#### Slow tokenizer
* written in Python inside the Transformers library
#### Fast tokenizer
* provided through the Tokenizers library
* written in Rust
* so it works much faster


### The most important feature of fast tokenizers
The most important power of a fast tokenizer is this:
It always keeps track of which part of the original text each final token came from.
This feature is called offset mapping.

Because of this, we can know:
* which token came from which word
* which character in the original text belongs to which token
* which token corresponds to which character range in the text


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
example="My name is Sakhawat and I work at Google in USA"
encoding = tokenizer(example)

print(type(encoding))

<class 'transformers.tokenization_utils_base.BatchEncoding'>


This means the tokenizer returned a BatchEncoding object.

### How to check whether it is a fast tokenizer
There are two ways to check.
#### 1) From the tokenizer


In [5]:
tokenizer.is_fast

True

#### 2) From the encoding

In [6]:
encoding.is_fast

True

### What can we do with a fast tokenizer?
#### 1) We can directly see the tokens


In [7]:
encoding.tokens()

['[CLS]',
 'My',
 'name',
 'is',
 'Sa',
 '##kha',
 '##wat',
 'and',
 'I',
 'work',
 'at',
 'Google',
 'in',
 'USA',
 '[SEP]']

#### 2) We can know which token came from which word


In [8]:
encoding.word_ids()

[None, 0, 1, 2, 3, 3, 3, 4, 5, 6, 7, 8, 9, 10, None]

This means:
* [CLS] and [SEP] are special tokens, so their value is None
* the rest show the index of the word each token came from


### What counts as a “word” is not always the same
What a word is depends on the tokenizer.
Example:
I'll
Is it one word or two words?
That depends on the tokenizer’s pre-tokenization rule.

### Some tokenizers
* split only on spaces
* then "I'll" is treated as one word
### Some tokenizers
* split on punctuation as well
* then "I'll" is treated as two words

### Getting character positions from a word or token
With a fast tokenizer, we can also get the character range from the original text.
Useful methods:
* word_to_chars()
* token_to_chars()
* char_to_word()
* char_to_token()


In [12]:
start,end = encoding.word_to_chars(3)
example[start:end]

'Sakhawat'

# Inside the token-classification pipeline

### What is NER?
NER is a task where we identify which parts of a text refer to entities such as:
* Person
* Location
* Organization
Examples:
Sakhawat → Person
Google → Organization
Chittagong → Location


### What does pipeline() do?
we saw that pipeline() combines three steps to get predictions from raw text:
#### 1) Tokenization
Split the text into tokens.
#### 2) Passing through the model
Feed the tokenized input into the model.
#### 3) Post-processing
Convert the model’s raw output into a readable result.
The text says:
* the first two steps are the same as in other pipelines
* but the post-processing in a token-classification pipeline is a bit more complex
This is because it is not enough just to get predictions; we also need to interpret the tokens correctly as entities.
#### Getting the basic result with the pipeline
First, a token classification pipeline is created:


In [15]:
from transformers import pipeline

token_classifier = pipeline("token-classification")
token_classifier("My name is sakhawat and i work at google in chittagong")


#The default model here is:
#dbmdz/bert-large-cased-finetuned-conll03-english
#This model performs NER on sentences.

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


[{'entity': 'I-PER',
  'score': np.float32(0.9892491),
  'index': 4,
  'word': 'sa',
  'start': 11,
  'end': 13},
 {'entity': 'I-PER',
  'score': np.float32(0.90338945),
  'index': 5,
  'word': '##kha',
  'start': 13,
  'end': 16},
 {'entity': 'I-PER',
  'score': np.float32(0.92671037),
  'index': 6,
  'word': '##wat',
  'start': 16,
  'end': 19}]

#### What do the output fields mean?
Each item contains several pieces of information:

* entity
What kind of entity it is
I-PER = part of a person entity
I-ORG = part of an organization entity
I-LOC = part of a location entity

* score
How confident the model is

* index
The token position

* word
The token itself

* start, end
The character positions in the original text


#### Grouping the tokens together
For this, we use aggregation_strategy="simple" in the pipeline:

In [17]:
token_classifier = pipeline("token-classification",aggregation_strategy="simple")
token_classifier("My name is sakhawat and i work at google in chittagong")

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


[{'entity_group': 'PER',
  'score': np.float32(0.9397829),
  'word': 'sakhawat',
  'start': 11,
  'end': 19}]

### What is aggregation_strategy?
It decides how to compute the final score when multiple tokens form one entity.
1) "simple"

Take the mean of the scores of all tokens in the entity.

Other available strategies
2) "first"
Use the score of the first token in the entity.

3) "max"
Use the maximum score among all tokens in the entity.

4) "average"
Average the scores of the words that make up the entity.
This is close to "simple", but calculated a little differently, especially when the entity has multiple words.

# From inputs to predictions

This section shows how the model produces predictions from raw text.
The main idea is:
* tokenize the text
* send the tokens to the model
* get probabilities and predictions from the model’s output logits
* then understand which token belongs to which entity


#### Step 1: Load the tokenizer and model


In [25]:
from transformers import AutoTokenizer,AutoModelForTokenClassification

model_checkpoint =  "dbmdz/bert-large-cased-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint,aggregation_strategy="simple")
model=AutoModelForTokenClassification.from_pretrained(model_checkpoint)

example = "My name is sakhawat and i work at google in chittagong"
inputs = tokenizer(example,return_tensors="pt")
outputs = model(**inputs)

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


### What happened here?
* AutoTokenizer converted the text into tokens
* AutoModelForTokenClassification loaded the NER model
* inputs = tokenizer(...) converted the text into the model’s input format
* outputs = model(**inputs) made the model produce predictions


In [26]:
inputs

{'input_ids': tensor([[  101,  1422,  1271,  1110, 21718, 14457, 20543,  1105,   178,  1250,
          1120,  1301,  8032,  1513,  1107, 22572,  9642, 19867,  1403,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [27]:
outputs

TokenClassifierOutput(loss=None, logits=tensor([[[10.1301, -2.3834, -1.3558, -2.1374, -1.3698, -2.0145, -0.6304,
          -2.0692,  0.7883],
         [10.4848, -2.2623, -1.7387, -2.4677, -0.3168, -2.0272,  0.2781,
          -2.2433, -0.3847],
         [10.9196, -2.3751, -1.3292, -2.4041, -0.4428, -2.0659, -0.2371,
          -1.9994, -0.2178],
         [11.0637, -2.2280, -1.7104, -2.1673, -0.9977, -1.7561, -0.0324,
          -1.8428, -0.2710],
         [ 1.2937, -2.4269, -1.2206, -2.8168,  6.4982, -3.1169, -0.1678,
          -2.6329,  0.7722],
         [ 2.2598, -2.8866, -0.4446, -2.4363,  4.7924, -3.6499,  0.3405,
          -3.1528,  0.0551],
         [ 2.7905, -2.5575, -1.2905, -3.4617,  5.5049, -3.0518,  0.5351,
          -3.1279, -0.0247],
         [10.8538, -2.1027, -1.6626, -2.1441, -1.3920, -1.6601,  0.1089,
          -1.7898, -0.2751],
         [ 9.5783, -1.8745, -1.5720, -2.2761, -0.3914, -2.1656,  0.7848,
          -1.5909, -0.5980],
         [11.0552, -2.0931, -1.4812, -2.56

### Step 2: Understand the output shape


In [28]:
print(inputs["input_ids"].shape)

torch.Size([1, 20])


This means:
batch size = 1
total tokens = 20
So there is 1 sentence, and it was split into 20 tokens.


In [32]:
print(outputs.logits.shape)

torch.Size([1, 20, 9])


This means:
1 sentence
20 tokens
9 label scores for each token
In other words, the model gives 9 possible class scores for each token.


# Step 3: Get probabilities and predictions from logits


In [36]:
import torch

probabilites = torch.nn.functional.softmax(outputs.logits,dim=-1)[0].tolist()
predictions = outputs.logits.argmax(dim=-1)[0].tolist()

In [38]:
print(probabilites)

[[0.9998519420623779, 3.676118694784236e-06, 1.0273134648741689e-05, 4.701446414401289e-06, 1.012982102110982e-05, 5.316504484653706e-06, 2.12198883673409e-05, 5.033492925576866e-06, 8.767411054577678e-05], [0.9999067783355713, 2.91037008537387e-06, 4.913004431728041e-06, 2.3701702502876287e-06, 2.036539081018418e-05, 3.681860562210204e-06, 3.691967867780477e-05, 2.9662046472367365e-06, 1.902779877127614e-05], [0.9999465942382812, 1.6832267419886193e-06, 4.790532784682e-06, 1.6351948488591006e-06, 1.1623516911640763e-05, 2.293254738106043e-06, 1.42785165735404e-05, 2.450858573865844e-06, 1.4556871974491514e-05], [0.9999556541442871, 1.6883207081264118e-06, 2.8329327506071422e-06, 1.7939920553544653e-06, 5.7781303439696785e-06, 2.706605755520286e-06, 1.5171241102507338e-05, 2.481725232428289e-06, 1.1950352927669883e-05], [0.005432643927633762, 0.00013158620276954025, 0.0004396215663291514, 8.9095818111673e-05, 0.9892491102218628, 6.599869084311649e-05, 0.0012597939930856228, 0.000107083

In [40]:
# Here we compute the probabilites for each tokens

In [41]:
print(predictions)

[0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## What is happening here?
### What are logits?
The model does not directly output the final label. It first gives raw scores, which are called logits.
### What is softmax?
softmax() converts logits into probabilities.
That means:
* it tells us how likely each label is
### What is argmax?
argmax() picks the label index with the highest score.
That means:
* it decides which class will be the final prediction


# Step 4: Look at the label mapping


In [43]:
model.config.id2label

{0: 'O',
 1: 'B-MISC',
 2: 'I-MISC',
 3: 'B-PER',
 4: 'I-PER',
 5: 'B-ORG',
 6: 'I-ORG',
 7: 'B-LOC',
 8: 'I-LOC'}

### What do these labels mean?
O
This means the token is not part of any entity.
O = Outside

B-XXX
Beginning of an entity

I-XXX
Inside an entity


For example:

* B-PER = beginning of a Person entity
* I-PER = inside a Person entity
* B-ORG = beginning of an Organization entity
* I-ORG = part of an Organization entity
* B-LOC = beginning of a Location entity
* I-LOC = part of a Location entity

# Step 5: Keep only the useful tokens, excluding O

In [46]:
results = []
tokens = inputs.tokens()

for idx, pred in enumerate(predictions):
    label = model.config.id2label[pred]
    if label != "O":
        results.append(
            {"entity": label, "score": probabilites[idx][pred], "word": tokens[idx]}
        )

print(results)


[{'entity': 'I-PER', 'score': 0.9892491102218628, 'word': 'sa'}, {'entity': 'I-PER', 'score': 0.9033893346786499, 'word': '##kha'}, {'entity': 'I-PER', 'score': 0.9267103672027588, 'word': '##wat'}]


### What happened here?
We:
* looked at all tokens
* kept only the ones whose label was not O
* stored the entity, score, and word for each one
This gives us the token-level NER result.

### What is still missing?
At this point, each token has:
* a label
* a score
* a word
But we still do not have the start and end character positions in the original sentence.
The pipeline had provided that information as well.
To get that, we need offset mapping.

# Step 6: Get offset mapping


In [47]:
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
inputs_with_offsets["offset_mapping"]


[(0, 0),
 (0, 2),
 (3, 7),
 (8, 10),
 (11, 13),
 (13, 16),
 (16, 19),
 (20, 23),
 (24, 25),
 (26, 30),
 (31, 33),
 (34, 36),
 (36, 38),
 (38, 40),
 (41, 43),
 (44, 46),
 (46, 49),
 (49, 53),
 (53, 54),
 (0, 0)]

# Step 8: Add start and end positions to the final result


In [50]:
results = []
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
tokens = inputs_with_offsets.tokens()
offsets = inputs_with_offsets["offset_mapping"]

for idx, pred in enumerate(predictions):
    label = model.config.id2label[pred]
    if label != "O":
        start, end = offsets[idx]
        results.append(
            {
                "entity": label,
                "score": probabilites[idx][pred],
                "word": tokens[idx],
                "start": start,
                "end": end,
            }
        )

print(results)


[{'entity': 'I-PER', 'score': 0.9892491102218628, 'word': 'sa', 'start': 11, 'end': 13}, {'entity': 'I-PER', 'score': 0.9033893346786499, 'word': '##kha', 'start': 13, 'end': 16}, {'entity': 'I-PER', 'score': 0.9267103672027588, 'word': '##wat', 'start': 16, 'end': 19}]
